# Cartão de Transporte 🚍

Prática de Linguagens Formais e Autômatos — Python e Jupyter.

**Integrantes**
- Bruno dos Reis Ferreira Santos — RA 2564952
- Vinicius de Freitas Ferreira — RA 2373516
- Matheus Francisco — RA 2693112
- Matheus de Oliveira — RA 2369955

Neste notebook mostramos os dois validadores, o rastreamento de estados, a comparação com expressões regulares, a catraca e o scanner. São exemplos fictícios para estudar os conceitos da disciplina.


In [ ]:
# Ajuste de leitura do notebook (nao altera a interface do navegador).
from IPython.display import HTML, display
display(HTML("""
<style>
.jp-NotebookPanel .jp-RenderedMarkdown,
.jp-NotebookPanel .jp-RenderedHTMLCommon {font-size:13px;line-height:1.45;}
.jp-NotebookPanel .jp-RenderedMarkdown h1 {font-size:24px;line-height:1.3;}
.jp-NotebookPanel .jp-RenderedMarkdown h2 {font-size:19px;line-height:1.35;margin-top:16px;}
.jp-NotebookPanel .jp-RenderedMarkdown h3 {font-size:16px;line-height:1.35;margin-top:12px;}
.jp-NotebookPanel .jp-RenderedMarkdown h4 {font-size:14px;line-height:1.35;}
.jp-NotebookPanel .jp-RenderedHTMLCommon table {font-size:12.5px;}
.jp-NotebookPanel .jp-RenderedMarkdown p {margin:5px 0 8px;}
.jp-NotebookPanel .jp-RenderedMarkdown li {margin:2px 0;}
.jp-NotebookPanel .jp-OutputArea svg {max-width:100%;height:auto;}
</style>
"""))
print("Apresentacao compacta ativada (titulos, paragrafos e tabelas).")


## Preparação

A célula abaixo localiza a pasta do projeto e carrega os módulos usados nos exemplos.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, SVG

# Localiza src tanto a partir da raiz quanto de notebooks/.
# Tambem funciona quando o kernel inicia em outras subpastas do projeto.
atual = Path.cwd().resolve()
RAIZ = next(
    (p for p in (atual, *atual.parents)
     if (p / "src" / "r1_afds.py").is_file()),
    None,
)
if RAIZ is None:
    raise FileNotFoundError(
        "Pasta src nao encontrada. Abra o Jupyter no projeto que contem README.md, src/ e notebooks/."
    )
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.r1_afds import cartao, linha, CLASSES, classificar
print("Projeto carregado corretamente:", RAIZ.name)
print("Validador 1:", cartao.nome)
print("Validador 2:", linha.nome)


## 2. Alfabeto e classes de caracteres

Os símbolos de entrada são classificados em **seis classes disjuntas** para que cada caractere possua **uma única classificação**. Isso evita transições ambíguas.

| Classe | Correspondência |
|---|---|
| `ZERO` | `0` |
| `UM_A_QUATRO` | `1, 2, 3, 4` |
| `CINCO_A_NOVE` | `5, 6, 7, 8, 9` |
| `PONTO` | `.` |
| `HIFEN` | `-` |
| `OUTRO` | Qualquer outro caractere |

A união das três primeiras classes equivale ao conjunto `0–9`, mas elas continuam distintas internamente.

In [ ]:
print("Alfabeto Σ:", CLASSES)
print("Classificação de amostras:")
for caractere in ("0", "3", "8", ".", "-", "X", " "):
    print(f" {caractere!r:>5} -> {classificar(caractere)}")

## Validador de cartão

Formato utilizado: `NN.NN.NNNNNNNN-N` (`N` representa um dígito de 0 a 9).

No AFD, `Q = {q0, q1, ..., q16, qerro}`, `Σ` contém as classes de caracteres da tabela acima, `q0` é o estado inicial e `F = {q16}`. A função `δ` fica em `cartao.delta`. Uma entrada só é aceita quando termina exatamente no estado final.


In [ ]:
print("AFD:", cartao.nome)
print("Regras:", cartao.descricao_regras)
print("Q =", ", ".join(cartao.estados))
print("Σ =", cartao.alfabeto)
print("q0 =", cartao.inicial)
print("F =", sorted(cartao.finais))
print("Transições definidas:", len(cartao.delta), "(=", len(cartao.estados), "×", len(cartao.alfabeto), ")")

### Diagrama do cartão

As setas mostram o avanço esperado. Qualquer caractere fora do formato leva a `qerro`.

In [ ]:
display(SVG(filename=str(RAIZ / "diagramas" / "afd_cartao.svg")))

### Tabela de transições do cartão

`→` indica o estado inicial, `*` indica o estado final e `qerro` indica rejeição.

In [ ]:
tabela_cartao = cartao.tabela()
assert len(tabela_cartao) == len(cartao.estados)

### Exemplo de validação do cartão

In [ ]:
cartao.rodar("12.34.56789012-3")
print("\n--- Exemplo rejeitado ---")
cartao.rodar("12.34.56789012")

## Validador da linha de ônibus

Formato de exemplo: `ABCD-EF`, como `8700-10`. O primeiro dígito vai de 1 a 9 e o dígito depois do hífen vai de 1 a 4; os demais vão de 0 a 9.

Este AFD possui `Q = {q0, q1, ..., q7, qerro}`, `q0` inicial e `F = {q7}`. A função de transição está em `linha.delta`.


In [ ]:
print("AFD:", linha.nome)
print("Regras:", linha.descricao_regras)
print("Q =", ", ".join(linha.estados))
print("Σ =", linha.alfabeto)
print("q0 =", linha.inicial)
print("F =", sorted(linha.finais))
print("Transições definidas:", len(linha.delta), "(=", len(linha.estados), "×", len(linha.alfabeto), ")")

### Diagrama da linha

In [ ]:
display(SVG(filename=str(RAIZ / "diagramas" / "afd_linha.svg")))

### Tabela de transições da linha

In [ ]:
tabela_linha = linha.tabela()
assert len(tabela_linha) == len(linha.estados)

### Exemplo de validação da linha

In [ ]:
linha.rodar("8700-10")
print("\n--- Exemplo rejeitado ---")
linha.rodar("8700-50")

### Outros exemplos

Testamos entradas corretas, incompletas, com caracteres extras e a palavra vazia.

In [ ]:
casos = {
    "cartao": (
        cartao,
        ["12.34.56789012-3", "00.00.00000000-0", "99.99.99999999-9"],
        ["", "12.34.56789012", "12.34.56789012-3X", "12-34-56789012-3"],
    ),
    "linha": (
        linha,
        ["8700-10", "1234-49", "9000-21", "4321-30"],
        ["", "0700-10", "8700-50", "8700-00", "8700-10X"],
    ),
}

quantidade = 0
for nome, (afd, aceitas, rejeitadas) in casos.items():
    for palavra in aceitas:
        assert afd.aceita(palavra), (nome, "deveria aceitar", palavra)
        quantidade += 1
    for palavra in rejeitadas:
        assert not afd.aceita(palavra), (nome, "deveria rejeitar", palavra)
        quantidade += 1
    assert len(afd.delta) == len(afd.estados) * len(afd.alfabeto)
    print(f"{nome.upper():>8}: {len(aceitas)} aceitas; {len(rejeitadas)} rejeitadas; δ completa")

print(f"✔ Validadores: {quantidade} casos exemplificados sem falhas.")

## Rastreamento dos estados

Os exemplos abaixo mostram o caminho que cada entrada percorre, caractere por caractere, até ser aceita ou rejeitada.


In [ ]:
from html import escape
from IPython.display import HTML
from src.r2_rastreamento import rastrear_detalhado, rodar_r2

def apresentar_rastreio(afd, palavra):
    """Tabela HTML legivel, gerada diretamente do resultado formal."""
    r = rastrear_detalhado(afd, palavra)
    linhas = []
    for p in r.passos:
        permitido = ", ".join(p.esperadas) if p.esperadas else "—"
        cor = "#fff1f2" if p.destino == "qerro" else ("#f0fdf4" if p.destino in afd.finais else "#fff")
        linhas.append(
            f'<tr style="background:{cor}">'
            f'<td>{p.posicao}</td><td>{escape(p.origem)}</td>'
            f'<td><code>{escape(repr(p.caractere))}</code></td>'
            f'<td>{escape(p.classe)}</td><td>{escape(permitido)}</td>'
            f'<td><strong>{escape(p.destino)}</strong></td>'
            f'<td>{escape(p.explicacao)}</td></tr>'
        )
    if not linhas:
        linhas = ['<tr><td colspan="7">Nenhum caractere consumido: permaneceu no estado inicial.</td></tr>']
    rotulo = 'ACEITA' if r.aceita else 'REJEITADA'
    estilo = '#166534' if r.aceita else '#991b1b'
    display(HTML(
        '<div style="margin:14px 0;padding:16px;border:1px solid #cbd5e1;border-radius:9px">'
        f'<h3 style="margin-top:0">{escape(afd.nome)} — <code>{escape(repr(palavra))}</code></h3>'
        '<div style="overflow-x:auto"><table style="border-collapse:collapse;font-size:12px;width:100%">'
        '<thead><tr style="background:#e2e8f0;color:#17202a">'
        '<th>Passo</th><th>Origem</th><th>Lido</th><th>Classe</th><th>Permitidas</th><th>Destino</th><th>Explicação</th>'
        '</tr></thead><tbody>' + ''.join(linhas) + '</tbody></table></div>'
        f'<p><strong style="color:{estilo}">{rotulo}</strong> | '
        f'Estado final: <code>{escape(r.estado_final)}</code></p>'
        f'<p><strong>Por quê?</strong> {escape(r.motivo)}</p></div>'
    ))
    return r

print("Funções de rastreamento prontas para os exemplos.")

### Cartão aceito — `12.34.56789012-3`

In [ ]:
r_cartao_valido = apresentar_rastreio(cartao, "12.34.56789012-3")
assert r_cartao_valido.aceita and r_cartao_valido.estado_final == "q16"
assert len(r_cartao_valido.passos) == 16

### Cartão rejeitado — `12.34.56789012-3X`

In [ ]:
r_cartao_invalido = apresentar_rastreio(cartao, "12.34.56789012-3X")
assert not r_cartao_invalido.aceita
assert r_cartao_invalido.passos[16].origem == "q16"
assert r_cartao_invalido.estado_final == "qerro"

### Linha aceita — `8700-10`

In [ ]:
r_linha_valida = apresentar_rastreio(linha, "8700-10")
assert r_linha_valida.aceita and r_linha_valida.estado_final == "q7"
assert len(r_linha_valida.passos) == 7

### Linha rejeitada — `8700-50`

In [ ]:
r_linha_invalida = apresentar_rastreio(linha, "8700-50")
assert not r_linha_invalida.aceita
assert r_linha_invalida.passos[5].destino == "qerro"
assert r_linha_invalida.passos[6].origem == "qerro"

### Entradas incompletas ou vazias

Mesmo após encontrar um caractere inválido, o AFD continua lendo o texto, mantendo o estado `qerro`.

In [ ]:
bordas = [
    (cartao, "12.34.56789012", "cartão incompleto"),
    (linha, "8700-1", "linha incompleta"),
    (cartao, "", "cartão: palavra vazia"),
    (linha, "", "linha: palavra vazia"),
    (linha, "8700-50XX", "erro após caractere inválido"),
]
for afd, palavra, titulo in bordas:
    r = rastrear_detalhado(afd, palavra)
    assert not r.aceita
    assert len(r.passos) == len(palavra)
    print(f"{titulo}: {palavra!r} -> {r.estado_final} | {r.motivo}")

### Rastreamento no console

Outra forma de acompanhar as transições de cada exemplo.

In [ ]:
for afd, palavra in [
    (cartao, "12.34.56789012-3"),
    (cartao, "12.34.56789012-3X"),
    (linha, "8700-10"),
    (linha, "8700-50"),
]:
    resultado = rodar_r2(afd, palavra)
    assert resultado.aceita == afd.aceita(palavra)
    print("\n" + "=" * 100 + "\n")

## Comparação com expressões regulares

Usamos `re.fullmatch()` para conferir a entrada inteira e comparar o resultado com os AFDs.

| Entrada | Expressão regular |
|---|---|
| Cartão | `[0-9]{2}\.[0-9]{2}\.[0-9]{8}-[0-9]` |
| Linha | `[1-9][0-9]{3}-[1-4][0-9]` |


In [ ]:
from html import escape
from IPython.display import display, HTML
from src.r3_equivalencia import (
    REGEX_CARTAO, REGEX_LINHA, CASOS_CARTAO, CASOS_LINHA,
    aceita_regex, executar_bateria, imprimir_bateria, testar_estresse,
)

print("AFD Cartão | RegEx:", REGEX_CARTAO.pattern)
print("AFD Linha  | RegEx:", REGEX_LINHA.pattern)
print("Exemplo 1:", "12.34.56789012-3", "->", aceita_regex(REGEX_CARTAO, "12.34.56789012-3"))
print("Exemplo 2:", "8700-50", "->", aceita_regex(REGEX_LINHA, "8700-50"))


### Exemplos de cartões

A tabela compara os resultados esperados, os AFDs e as expressões regulares.

In [ ]:
def mostrar_bateria_r3(bateria):
    """Tabela legível no Jupyter; sem dependência de pandas."""
    def status(valor):
        return "ACEITA" if valor else "REJEITA"

    cabecalho = ("#", "Entrada", "Esperado", "AFD", "RegEx", "Concordam?", "Correto?", "Motivo")
    th = "".join(f"<th style='text-align:left;padding:6px;border-bottom:2px solid #6b7280'>{escape(c)}</th>" for c in cabecalho)
    tr = []
    for i, caso in enumerate(bateria.resultados, 1):
        valores = [str(i), '"" (vazia)' if caso.entrada == "" else repr(caso.entrada),
                   status(caso.esperado), status(caso.aceita_afd), status(caso.aceita_regex),
                   "SIM" if not caso.divergencia else "NÃO",
                   "SIM" if not caso.falha_de_expectativa else "NÃO",
                   caso.justificativa]
        celulas = "".join(f"<td style='padding:6px;border-bottom:1px solid #7775'>{escape(v)}</td>" for v in valores)
        tr.append(f"<tr>{celulas}</tr>")
    tabela_html = f"<table style='border-collapse:collapse;width:100%;font-size:12.5px;line-height:1.35'><thead><tr>{th}</tr></thead><tbody>{''.join(tr)}</tbody></table>"
    display(HTML(f"<h3 style='font-size:16px'>{escape(bateria.nome)}</h3>" + tabela_html))
    print(f"RESUMO — {bateria.nome}: {bateria.total_aceitas} aceitas, "
          f"{bateria.total_rejeitadas} rejeitadas; "
          f"divergências={bateria.divergencias}; "
          f"erros de expectativa={bateria.falhas_de_expectativa}")

bateria_cartao = executar_bateria(cartao, REGEX_CARTAO, CASOS_CARTAO)
mostrar_bateria_r3(bateria_cartao)
assert bateria_cartao.aprovado, "Falha na equivalência ou no resultado esperado para cartão"
assert bateria_cartao.total_aceitas >= 5 and bateria_cartao.total_rejeitadas >= 5
assert any(r.entrada == "" and not r.esperado for r in bateria_cartao.resultados)


### Exemplos de linhas

Incluímos entradas aceitas e rejeitadas, inclusive a palavra vazia.

In [ ]:
bateria_linha = executar_bateria(linha, REGEX_LINHA, CASOS_LINHA)
mostrar_bateria_r3(bateria_linha)
assert bateria_linha.aprovado, "Falha na equivalência ou no resultado esperado para linha"
assert bateria_linha.total_aceitas >= 5 and bateria_linha.total_rejeitadas >= 5
assert any(r.entrada == "" and not r.esperado for r in bateria_linha.resultados)


### Comparação da entrada inteira

Veja por que `fullmatch()` é diferente de procurar apenas uma parte do texto.

In [ ]:
import re
exemplos = [
    ("cartão com texto extra", cartao, REGEX_CARTAO, "xx12.34.56789012-3yy"),
    ("cartão com newline", cartao, REGEX_CARTAO, "12.34.56789012-3\n"),
    ("linha com sufixo", linha, REGEX_LINHA, "8700-10X"),
    ("linha com Unicode", linha, REGEX_LINHA, "8７00-10"),
    ("palavra vazia", linha, REGEX_LINHA, ""),
]
for rotulo, afd, regex, entrada in exemplos:
    busca = re.search(regex, entrada) is not None
    total = aceita_regex(regex, entrada)
    resultado_afd = afd.aceita(entrada)
    print(f"{rotulo}: entrada={entrada!r} | search={busca} | fullmatch={total} | AFD={resultado_afd}")
    assert not total and not resultado_afd


## Simulação da catraca

O programa representa a passagem por estados da catraca. Cada evento permitido altera o estado; eventos fora da ordem são recusados sem alterar o estado atual.


In [ ]:
from src.r4_catraca import (
    Catraca, CENARIOS, DELTA, DESCRICOES_ESTADOS, ESTADOS, EVENTOS,
    TRANSICOES, verificar_modelo, executar_cenarios, texto_fluxo,
)
from IPython.display import display, SVG, HTML
from html import escape

verificar_modelo()
print("Máquina de estados da catraca")
print("Estado inicial:", Catraca().estado)
print("Estados:", len(ESTADOS), "| Transições:", len(TRANSICOES),
      "| Eventos distintos:", len(EVENTOS))
assert len(ESTADOS) >= 5 and len(TRANSICOES) >= 6
for estado in ESTADOS:
    print(f"{estado:<24} | {DESCRICOES_ESTADOS[estado]}")


### Diagrama da catraca

In [ ]:
display(SVG(filename=str(RAIZ / "diagramas" / "catraca_estados.svg")))

### Tabela dos eventos

Cada linha informa o estado de origem, o evento e o próximo estado.

In [ ]:
transicoes_impressas = Catraca().tabela()
assert len(transicoes_impressas) == 12
assert len(DELTA) == len(transicoes_impressas)
assert Catraca().permitidos("CARTAO_LIDO") == ("VALIDAR_CARTAO", "CANCELAR")


### Dois embarques válidos

Um embarque direto e outro com recarga simulada.

In [ ]:
def exibir_fluxo_r4(resultado):
    cabecalho = ["#", "Origem", "Evento", "Destino", "Resultado"]
    linhas = []
    for passo in resultado.passos:
        valores = [str(passo.numero), passo.origem, passo.evento, passo.destino,
                   "ACEITO" if passo.aceito else "RECUSADO"]
        fundo = "#dcfce7" if passo.aceito else "#fee2e2"
        tds = "".join("<td style='padding:7px;border-bottom:1px solid #e5e7eb'>" +
                      escape(valor) + "</td>" for valor in valores)
        linhas.append(f"<tr style='background:{fundo}'>{tds}</tr>")
        if not passo.aceito:
            mensagem = ("Recusa: " + passo.explicacao + " Eventos permitidos: " +
                        ", ".join(passo.permitidos))
            linhas.append("<tr><td colspan='5' style='padding:9px;background:#fff7ed;color:#7c2d12'>" +
                          escape(mensagem) + "</td></tr>")
    ths = "".join("<th style='background:#0f172a;color:white;padding:7px;text-align:left'>" +
                  escape(x) + "</th>" for x in cabecalho)
    resumo = (f"Estado final: {resultado.estado_final} | "
              f"Passagens: {resultado.passagens_registradas} | "
              f"Eventos: {len(resultado.passos)}/{resultado.eventos_solicitados}")
    display(HTML(f"<div style='font-family:Arial,sans-serif;color:#0f172a'>"
                 f"<h3>{escape(resultado.titulo)}</h3>"
                 f"<table style='width:100%;border-collapse:collapse;font-size:13px'>"
                 f"<tr>{ths}</tr>{''.join(linhas)}</table>"
                 f"<p><b>{escape(resumo)}</b></p></div>"))
    print("DECISÃO:", "FLUXO VÁLIDO" if resultado.completo else "FLUXO INVÁLIDO")

r4_resultados = executar_cenarios(mostrar=False)
assert len(r4_resultados) == 4
for fluxo in r4_resultados[:2]:
    exibir_fluxo_r4(fluxo)
    assert fluxo.viagem_concluida
    assert fluxo.passagens_registradas == 1


### Dois exemplos de eventos recusados

Os motivos da recusa e os próximos eventos possíveis aparecem no resultado.

In [ ]:
for fluxo in r4_resultados[2:]:
    exibir_fluxo_r4(fluxo)
    assert fluxo.contem_recusa and not fluxo.completo
    assert fluxo.passagens_registradas == 0
    recusa = fluxo.passos[-1]
    assert not recusa.aceito and recusa.destino == recusa.origem
    assert len(recusa.permitidos) > 0
    print("EVENTOS PERMITIDOS NO ESTADO:", ", ".join(recusa.permitidos))


### Outros testes da catraca

In [ ]:
catraca_teste = Catraca()
errado = catraca_teste.disparar("PULAR_CATRACA")
print("Evento desconhecido:", errado.explicacao)
assert not errado.aceito and catraca_teste.estado == "TRAVADA"
assert errado.permitidos == ("APROXIMAR_CARTAO",)

catraca_teste.disparar("APROXIMAR_CARTAO")
recusa = catraca_teste.disparar("INICIAR_GIRO")
print("\nGiro antecipado:", recusa.explicacao)
assert not recusa.aceito and catraca_teste.estado == "CARTAO_LIDO"
assert catraca_teste.disparar("VALIDAR_CARTAO").aceito
assert catraca_teste.estado == "VERIFICANDO_SALDO"
print("Recuperação após recusa: agora o estado é", catraca_teste.estado)

com_tempo = Catraca()
fluxo_tempo = com_tempo.executar((
    "APROXIMAR_CARTAO", "VALIDAR_CARTAO", "SALDO_SUFICIENTE", "TEMPO_ESGOTADO"))
assert fluxo_tempo.completo and fluxo_tempo.estado_final == "TRAVADA"
assert fluxo_tempo.passagens_registradas == 0
print("Expiração: voltou a TRAVADA, sem registrar passagem")


## Scanner de comandos

O scanner separa textos de exemplo em tokens, informando tipo, conteúdo, linha e coluna. Também mostra erros e continua a leitura.


In [ ]:
from src.r5_scanner import (
    analisar, imprimir_resultado, tabela_tokens, RESERVADAS, TIPOS_DE_TOKEN,
    EXEMPLO_VALIDO, EXEMPLO_COM_ERROS,
)
from IPython.display import display, HTML
from html import escape

assert len(TIPOS_DE_TOKEN) == 21 and len(RESERVADAS) == 5
print('Scanner: ', len(TIPOS_DE_TOKEN), 'tipos de token,', len(RESERVADAS), 'palavras reservadas')
print('Reservadas:', ', '.join(RESERVADAS))


### Tipos de token

A tabela abaixo vem das regras usadas pelo scanner.

In [ ]:
tabela_completa = tabela_tokens()
linhas_html = []
for tipo, descricao, exemplo in tabela_completa:
    linhas_html.append('<tr><td><code>'+escape(tipo)+'</code></td><td><code>'+escape(exemplo)+'</code></td><td>'+escape(descricao)+'</td></tr>')
display(HTML("<style>.r5table{width:100%;border-collapse:collapse;font-family:Arial,sans-serif;font-size:13px;}"
             ".r5table td,.r5table th{padding:7px 9px;border:1px solid #cbd5e1;text-align:left;}"
             ".r5table th{background:#1e293b;color:white}</style>"
             "<table class='r5table'><thead><tr><th>Tipo</th><th>Exemplo</th><th>Descrição</th></tr></thead>"
             '<tbody>'+''.join(linhas_html)+'</tbody></table>'))


### Exemplo de leitura sem erros

In [ ]:
print('PROGRAMA DE ENTRADA:')
print(EXEMPLO_VALIDO)
resultado_valido = analisar(EXEMPLO_VALIDO)
assert resultado_valido.aprovado and len(resultado_valido.erros) == 0
imprimir_resultado(resultado_valido)
assert [(t.tipo, t.linha, t.coluna) for t in resultado_valido.tokens if t.tipo == 'NUM_CARTAO'] == [('NUM_CARTAO', 2, 22)]


### Leitura do maior token

Por exemplo, `>=` é reconhecido como um único token, e `20.00` é um número real.

In [ ]:
exemplo_munch = 'SALDO >= 20.00; SALDO > 20; SALDO <= 30; SALDO == 20; RECARGA_EXTRA 8700-10'
r_munch = analisar(exemplo_munch)
assert not r_munch.erros
print('Entrada:', exemplo_munch)
print('Tokens:', ' | '.join(t.tipo + ':' + t.lexema for t in r_munch.tokens))
assert any(t.tipo == 'MAIOR_IGUAL' and t.lexema == '>=' for t in r_munch.tokens)
assert any(t.tipo == 'MENOR_IGUAL' and t.lexema == '<=' for t in r_munch.tokens)
assert any(t.tipo == 'IGUAL_IGUAL' and t.lexema == '==' for t in r_munch.tokens)
assert any(t.tipo == 'REAL' and t.lexema == '20.00' for t in r_munch.tokens)
assert any(t.tipo == 'IDENTIFICADOR' and t.lexema == 'RECARGA_EXTRA' for t in r_munch.tokens)
assert any(t.tipo == 'NUM_LINHA' and t.lexema == '8700-10' for t in r_munch.tokens)
print('Maximal Munch comprovado nos exemplos.')


### Espaços, comentários e localização

In [ ]:
programa_com_comentario = ('# SALDO >= 100 dentro do comentario\r\n'
                         '\tRECARGA 20.00 // ignorar tokens 123 !=\r\n'
                         'EMBARQUE 8700-10 07:42')
r_comentario = analisar(programa_com_comentario)
assert r_comentario.aprovado
assert [(t.tipo, t.linha, t.coluna) for t in r_comentario.tokens] == [
    ('RECARGA', 2, 2), ('REAL', 2, 10),
    ('EMBARQUE', 3, 1), ('NUM_LINHA', 3, 10), ('HORARIO', 3, 18)]
print('Comentários foram ignorados e as posições dos 5 tokens são:')
for t in r_comentario.tokens:
    print(f'[{t.linha}:{t.coluna}] {t.tipo:15} {t.lexema}')


### Exemplo com erros

O scanner mostra onde ocorreu cada erro e continua analisando as próximas linhas.

In [ ]:
print('PROGRAMA COM ERROS:')
print(EXEMPLO_COM_ERROS)
resultado_com_erros = analisar(EXEMPLO_COM_ERROS)
imprimir_resultado(resultado_com_erros)
assert [(e.linha, e.coluna) for e in resultado_com_erros.erros] == [(1, 22), (2, 10), (2, 18)]
assert len(resultado_com_erros.erros) == 3
assert [t.tipo for t in resultado_com_erros.tokens[-4:]] == ['RECARGA', 'REAL', 'CARTAO', 'NUM_CARTAO']
assert resultado_com_erros.tokens[-1].linha == 3
for erro in resultado_com_erros.erros:
    display(HTML('<div style="padding:8px;border-left:4px solid #dc2626;background:#fff4ed;'
                 'color:#6b2415;margin:6px 0"><b>Erro lexical [linha '+str(erro.linha)+
                 ', coluna '+str(erro.coluna)+']</b>: <code>'+escape(erro.lexema)+'</code>'
                 '<div>'+escape(erro.mensagem)+'</div><div><b>Dica:</b> '+
                 escape(erro.dica)+'</div></div>'))
print('RECUPERAÇÃO: o scanner reconheceu o cartão válido na linha 3, mesmo após 3 erros.')


### Conferência dos formatos

Cartões e linhas reconhecidos pelo scanner seguem as mesmas regras dos validadores.

In [ ]:
from src.r3_equivalencia import CASOS_CARTAO, CASOS_LINHA
for casos, tipo in ((CASOS_CARTAO, 'NUM_CARTAO'), (CASOS_LINHA, 'NUM_LINHA')):
    for caso in casos:
        if caso.esperado:
            retorno = analisar(caso.palavra)
            assert retorno.aprovado, caso.palavra
            assert len(retorno.tokens) == 1, caso.palavra
            assert retorno.tokens[0].tipo == tipo, caso.palavra
print('16 exemplos aceitos pelos AFDs/RegEx também foram reconhecidos pelo scanner (8 cartões e 8 linhas).')


## Painel interativo

O painel reúne três abas: **Validador**, **Catraca** e **Scanner**. É preciso executá-lo no Jupyter com a biblioteca `ipywidgets` instalada.


### Abrir as três abas

Depois de executar esta célula, experimente os exemplos nas abas.

In [ ]:
# Painel independente da ordem das celulas.
# Esta celula pode ser executada sem as anteriores.
from pathlib import Path
import sys
from IPython.display import display, HTML

pasta_atual = Path.cwd().resolve()
raiz_painel = next(
    (p for p in (pasta_atual, *pasta_atual.parents)
     if (p / "src" / "r6_interface.py").is_file()),
    None,
)
if raiz_painel is None:
    raise FileNotFoundError(
        "Nao encontrei src/r6_interface.py. Confira se o ZIP foi extraido inteiro "
        "e inicie o Jupyter na pasta do projeto (onde ficam src e notebooks)."
    )
if str(raiz_painel) not in sys.path:
    sys.path.insert(0, str(raiz_painel))
RAIZ = raiz_painel
print("Raiz do projeto localizada:", RAIZ)

try:
    from src.r6_interface import LaboratorioR6
    laboratorio = LaboratorioR6()
except ModuleNotFoundError as erro:
    if "ipywidgets" not in str(erro):
        raise
    laboratorio = None
    display(HTML("""<div style="padding:12px;border-left:4px solid #bb7626;">
        <strong>Interface indisponivel neste kernel:</strong> falta instalar
        <code>ipywidgets</code>.<br>
        Instale no MESMO Python do Jupyter, reinicie o kernel e execute a celula novamente.
        </div>"""))
else:
    display(laboratorio.widget)
    print("Painel carregado: 3 abas (AFD, Catraca e Scanner).")


### Exemplo do validador

Estas saídas também podem ser consultadas no GitHub.

In [ ]:
from src.r6_interface import avaliar_entrada, html_validacao
from IPython.display import display, HTML
for nome, palavra in (("cartao", "12.34.56789012-3"),
                      ("cartao", "12.34.56789012-3X"),
                      ("linha", "8700-10"),
                      ("linha", "8700-50"),
                      ("cartao", "")):
    display(HTML(html_validacao(avaliar_entrada(nome, palavra))))
print("✅ Cinco relatórios AFD exibidos, incluindo palavra vazia e duas rejeições.")


### Exemplo do scanner com erros

In [ ]:
from src.r6_interface import html_scanner
scanner_r6 = analisar(EXEMPLO_COM_ERROS)
assert len(scanner_r6.erros) == 3
assert any(t.tipo == "NUM_CARTAO" and t.linha == 3 for t in scanner_r6.tokens)
display(HTML(html_scanner(scanner_r6)))
print("✅ Scanner: três erros diagnosticados; leitura continuou até a última linha.")


### Exemplo de evento recusado na catraca

In [ ]:
from src.r6_interface import html_catraca
from src.r4_catraca import Catraca
catraca_r6 = Catraca()
passo_recusado = catraca_r6.disparar("INICIAR_GIRO")
assert not passo_recusado.aceito
assert catraca_r6.estado == "TRAVADA"
display(HTML(html_catraca(catraca_r6, ultimo=passo_recusado)))
print("✅ Catraca: evento recusado sem alterar o estado.")
